# 01 — Prise en main de Kili

Ce notebook est une **visite guidée**. Les scripts de `examples/` restent la source de vérité : ils sont testés, versionnés et exécutables en une commande.

Objectifs :

1. se connecter à l'instance Kili on-premise ;
2. créer un projet minimal ;
3. comprendre l'anatomie d'un `json_interface`.

> ⚠️ Les cellules qui appellent Kili nécessitent un `.env` valide et une instance joignable. Les cellules de construction d'interface, elles, tournent hors ligne.

## 1. Configuration

Les identifiants ne sont jamais écrits dans le code : ils viennent du fichier `.env` à la racine, lu par pydantic-settings.

In [ ]:
from kili_examples.config import settings

# On n'affiche jamais la clé elle-même, seulement sa présence.
print("Endpoint :", settings.kili_api_endpoint)
print("Clé API renseignée :", bool(settings.kili_api_key))

## 2. Anatomie d'un `json_interface`

Une interface Kili est un dictionnaire JSON : `{"jobs": {...}}`. Chaque *job* décrit une tâche d'annotation. Construisons le plus simple possible.

In [ ]:
import json

from kili_examples.interfaces import (
    build_category,
    build_classification_job,
    build_json_interface,
)

jobs = {
    "TYPE_SINISTRE": build_classification_job(
        instruction="Quel est le type de sinistre ?",
        categories={
            "SINISTRE_AUTO": build_category("Sinistre auto"),
            "DEGAT_DES_EAUX": build_category("Dégât des eaux"),
        },
        input_type="radio",
    )
}
json_interface = build_json_interface(jobs)
print(json.dumps(json_interface, indent=2, ensure_ascii=False))

### Les clés à retenir

| Clé | Rôle |
| --- | --- |
| `mlTask` | le type d'annotation (`CLASSIFICATION`, `OBJECT_DETECTION`, `NAMED_ENTITIES_RECOGNITION`, `TRANSCRIPTION`) |
| `content.categories` | les catégories proposées, indexées par leur **clé métier** |
| `content.input` | le widget : `radio` (une réponse), `checkbox` (plusieurs), `dropdown` (liste longue) |
| `instruction` | la consigne affichée à l'annotateur |
| `required` | `1` si le job doit être rempli, `0` sinon |
| `isChild` | `True` pour un sous-job conditionnel |

Point important : la **clé** de la catégorie (`SINISTRE_AUTO`) est ce qu'on retrouve dans les exports et dans les prédictions ; le champ `name` n'est que le libellé affiché.

## 3. Rendre un job conditionnel

Un sous-job apparaît quand une catégorie précise est sélectionnée. Deux choses à faire : déclarer `children` sur la catégorie parente, et marquer le sous-job `isChild=True`.

In [ ]:
jobs_hierarchiques = {
    "TYPE_SINISTRE": build_classification_job(
        instruction="Quel est le type de sinistre ?",
        categories={
            "SINISTRE_AUTO": build_category(
                "Sinistre auto",
                children=["SOUS_TYPE_AUTO"],  # <- déclencheur
            ),
            "DEGAT_DES_EAUX": build_category("Dégât des eaux"),
        },
    ),
    "SOUS_TYPE_AUTO": build_classification_job(
        instruction="Nature du sinistre auto ?",
        categories={
            "COLLISION": build_category("Collision"),
            "STATIONNEMENT": build_category("Stationnement"),
        },
        required=False,
        is_child=True,  # <- ne s'affiche jamais seul
    ),
}
print(
    json.dumps(
        build_json_interface(jobs_hierarchiques),
        indent=2,
        ensure_ascii=False,
    )
)

## 4. Créer le projet

⚠️ **Cette cellule écrit sur l'instance Kili.** Elle crée un vrai projet : à ne lancer que sur une instance de test.

In [ ]:
from kili_examples.client import get_kili

kili = get_kili()
projet = kili.create_project(
    title="Notebook 01 - projet de decouverte",
    description="Projet jetable créé depuis le notebook.",
    input_type="TEXT",
    json_interface=json_interface,
)
print("project_id =", projet["id"])

Notez le `project_id` affiché : il permet de retrouver le projet dans l'interface web, et de rejouer les scripts avec `--project-id <id>` sans recréer de projet.

**Suite** : `02_cycle_complet.ipynb` déroule les quatre étapes (créer, importer, prédire, exporter).